# Empirical Evaluation of Prompting, Retrieval-Augmented Generation, and Parameter-Efficient Fine-Tuning for Medical Question Answering

**Course:** COMP5801H - Generative AI and Large Language Models

**Institution:** Carleton University, Winter 2026

**Author:** Dhanshree Kunjalkumar Soni — 101384658

**Project Option:** Option A - Empirical Evaluation

---

## Project Overview

This notebook contains the complete implementation for my course project, which empirically compares three strategies for adapting a large language model to domain-specific medical question answering: prompt engineering, retrieval-augmented generation, and parameter-efficient fine-tuning. All five experimental conditions use Mistral-7B-Instruct-v0.2 as the base model to ensure a fair, controlled comparison. Experiments are conducted on the MedQA (USMLE) benchmark, a publicly available dataset of four-choice questions from the United States Medical Licensing Examination.

The central research question is: under equal compute and data budgets, which adaptation strategy provides the best accuracy-efficiency trade-off for domain-specific medical question answering?

## Notebook Structure

| Section | Content |
|---------|---------|
| Section 1 | Environment Setup and Library Installation |
| Section 2 | Dataset Loading and Preprocessing |
| Section 3 | Loading the Base Model |
| Section 4 | Prompting Baselines: Zero-Shot, Few-Shot, Chain-of-Thought |
| Section 5 | Retrieval-Augmented Generation — FAISS pipeline, ablation k=1,3,5 |
| Section 6 | QLoRA Fine-Tuning |
| Section 7 | Results, Cost Analysis and Error Analysis |
| Section 8 | Conclusion |

# Section 1: Environment Setup

Before anything else, I need to install all the libraries this project depends on. This includes the core transformer and quantization libraries for loading Mistral-7B, the dataset library to pull MedQA from HuggingFace, evaluation libraries for BERTScore and F1, FAISS for the retrieval pipeline, and PEFT/TRL for QLoRA fine-tuning. I am installing everything upfront so there are no interruptions later in the notebook.

In [2]:
# Install all required libraries
!pip install -q transformers accelerate bitsandbytes
!pip install -q datasets evaluate
!pip install -q bert-score rouge-score
!pip install -q faiss-cpu sentence-transformers
!pip install -q peft trl
!pip install -q pandas numpy matplotlib seaborn tqdm
!pip install -q scikit-learn

print("All libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.4 MB/s eta 0:00:00
All libraries installed successfully.


In [3]:
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U transformers accelerate
import importlib
import bitsandbytes
importlib.reload(bitsandbytes)
print("bitsandbytes updated successfully.")
print("Version:", bitsandbytes.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.8 MB/s eta 0:00:00
bitsandbytes updated successfully.
Version: 0.49.2


# Section 2: Dataset Loading and Preprocessing

In this section I load the MedQA (USMLE) dataset, verify the GPU is available for model inference, and preprocess the data into a clean format that all five experimental conditions will use consistently. I also do a quick inspection of the data structure to make sure everything looks correct before building anything on top of it.

## 2.1 Verifying the GPU Environment

Before loading any models or data, I verify that the GPU is available and check its memory capacity. This matters because Mistral-7B-Instruct requires at least 8GB of VRAM even with 4-bit quantization, so confirming the hardware upfront avoids surprises later.

In [4]:
import torch
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

# Verify GPU
print("=== GPU STATUS ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

=== GPU STATUS ===
CUDA available: True
GPU count: 2
GPU name: Tesla T4
GPU memory: 15.6 GB


## 2.2 Loading the MedQA Dataset

I use the GBaker/MedQA-USMLE-4-options version of MedQA from HuggingFace, which is a clean Parquet-based version of the original dataset containing questions from the United States Medical Licensing Examination. Each question has exactly four answer choices (A, B, C, D) which makes accuracy a clean and unambiguous primary metric.

Note: this dataset does not include a validation split, only train and test. I handle this in the next cell by carving out a validation set from the training data manually.

In [5]:
from datasets import load_dataset

print("Loading MedQA dataset...")
dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

print("\nDataset structure:")
print(dataset)

print("\nTraining samples:", len(dataset['train']))
print("Test samples:", len(dataset['test']))

# Inspect one sample to understand the structure
print("\n=== SAMPLE QUESTION ===")
sample = dataset['train'][0]
for key, value in sample.items():
    print(f"{key}: {value}")

Loading MedQA dataset...


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]


Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})

Training samples: 10178
Test samples: 1273

=== SAMPLE QUESTION ===
question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?


## 2.3 Creating the Validation Split

The dataset comes with a train split and a test split but no separate validation split. I carve out 10% of the training data as a validation set using a fixed random seed of 42 to ensure reproducibility. The validation set is what I use to compare all five techniques against each other, keeping the test set untouched for final evaluation.

In [6]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

# This dataset has train and test only, so we carve out a validation set
# from training data using an 90/10 split — standard practice
train_val = dataset['train'].train_test_split(test_size=0.1, seed=42)

train_data = train_val['train']
val_data   = train_val['test']
test_data  = dataset['test']

print("Split sizes:")
print(f"  Train:      {len(train_data)} samples")
print(f"  Validation: {len(val_data)} samples")
print(f"  Test:       {len(test_data)} samples")

# Inspect one sample to understand the exact structure
print("\n=== SAMPLE QUESTION ===")
sample = train_data[0]
print("Question:", sample['question'])
print("Options:", sample['options'])
print("Answer:", sample['answer'])
print("Answer idx:", sample['answer_idx'])

Split sizes:
  Train:      9160 samples
  Validation: 1018 samples
  Test:       1273 samples

=== SAMPLE QUESTION ===
Question: A 60-year-old man comes to the physician because of flank pain, rash, and blood-tinged urine for 1 day. Two months ago, he was started on hydrochlorothiazide for hypertension. He takes acetaminophen for back pain. Examination shows a generalized, diffuse maculopapular rash. Serum studies show a creatinine concentration of 3.0 mg/dL. Renal ultrasonography shows no abnormalities. Which of the following findings is most likely to be observed in this patient?
Options: {'A': 'Dermal IgA deposition on skin biopsy', 'B': 'Crescent-shape extracapillary cell proliferation', 'C': 'Mesangial IgA deposits on renal biopsy', 'D': 'Urinary eosinophils'}
Answer: Urinary eosinophils
Answer idx: D


## 2.4 Preprocessing into DataFrames

I convert the raw HuggingFace dataset into clean pandas DataFrames with one column per answer option. This consistent structure is used across all five experimental conditions so the formatting logic stays the same throughout the notebook. I also save the three splits as CSV files so they can be reloaded quickly in later sections without re-downloading the dataset.

In [7]:
def build_dataframe(data):
    rows = []
    for item in data:
        options = item['options']
        rows.append({
            'question': item['question'],
            'option_A': options.get('A', ''),
            'option_B': options.get('B', ''),
            'option_C': options.get('C', ''),
            'option_D': options.get('D', ''),
            'answer':   item['answer_idx'],
            'answer_text': item['answer']
        })
    return pd.DataFrame(rows)

train_df = build_dataframe(train_data)
val_df   = build_dataframe(val_data)
test_df  = build_dataframe(test_data)

# Save to CSV so every section of the notebook can reload them
# without re-downloading the dataset
train_df.to_csv('/kaggle/working/train.csv', index=False)
val_df.to_csv('/kaggle/working/val.csv', index=False)
test_df.to_csv('/kaggle/working/test.csv', index=False)

print("DataFrames built and saved.")
print(f"Train:      {len(train_df)} rows")
print(f"Validation: {len(val_df)} rows")
print(f"Test:       {len(test_df)} rows")

print("\n=== PREVIEW (first 2 rows) ===")
train_df.head(2)

DataFrames built and saved.
Train:      9160 rows
Validation: 1018 rows
Test:       1273 rows

=== PREVIEW (first 2 rows) ===


,question,option_A,option_B,option_C,option_D,answer,answer_text
0,A 60-year-old man comes to the physician becau...,Dermal IgA deposition on skin biopsy,Crescent-shape extracapillary cell proliferation,Mesangial IgA deposits on renal biopsy,Urinary eosinophils,D,Urinary eosinophils
1,A 53-year-old male presents to your office for...,Inhibits degradation of endogenous incretins,Inhibits alpha-glucosidases at the intestinal ...,Activates transcription of PPARs to increase p...,Increases secretion of insulin in response to ...,A,Inhibits degradation of endogenous incretins


# Section 3: Loading the Base Model

I use Mistral-7B-Instruct-v0.2 as the base model for all five experimental conditions in this project. Using the same model across every technique is a deliberate design choice, it ensures that any difference in performance is due to the adaptation strategy itself, not the underlying model.

Since a 7B parameter model is too large to fit in GPU memory at full precision, I load it in 4-bit quantization using BitsAndBytes. This reduces the memory footprint from roughly 28GB down to about 6-8GB while keeping performance very close to the full precision version. The quantization settings I use are standard for inference on consumer-grade GPUs.

## 3.1 Loading the Tokenizer and Model

In [8]:
!pip install -q -U bitsandbytes
import subprocess
result = subprocess.run(['pip', 'show', 'bitsandbytes'], capture_output=True, text=True)
print(result.stdout)

Name: bitsandbytes
Version: 0.49.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, torch
Required-by: 



In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import time

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
# nf4 is the recommended quantization type for inference
# double quantization further reduces memory usage
# bfloat16 is used for computation to maintain numerical stability
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print("Tokenizer loaded.")

print("\nLoading model in 4-bit quantization (this takes 2-3 minutes)...")
start = time.time()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()
elapsed = round(time.time() - start, 1)

print(f"\nModel loaded in {elapsed}s")
print(f"Memory footprint: {round(model.get_memory_footprint() / 1e9, 2)} GB")
print(f"Device map: {model.hf_device_map}")

Loading tokenizer...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


Model loaded in 55.0s
Memory footprint: 4.01 GB
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## 3.2 Sanity Check

Before running any experiments, I do a quick sanity check to confirm the model can generate coherent responses to a medical question. I also define the core inference function that all three prompting techniques will use throughout Section 4. The function records inference latency for every call, which feeds directly into the cost analysis in Section 6.

In [10]:
import time

def generate_response(prompt, max_new_tokens=200):
    """
    Core inference function used by all experimental conditions.
    Returns the model response and the time taken in milliseconds.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    latency_ms = round((time.time() - start) * 1000, 1)

    # Decode only the newly generated tokens, not the input prompt
    input_length = inputs['input_ids'].shape[1]
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response, latency_ms


# Sanity check with a straightforward medical question
test_prompt = """<s>[INST] A 45-year-old patient presents with chest pain radiating to the left arm and diaphoresis. What is the most likely diagnosis?

A) Pneumonia
B) Myocardial infarction
C) Acid reflux
D) Pulmonary embolism

Answer with only the letter of the correct option. [/INST]"""

response, latency = generate_response(test_prompt, max_new_tokens=10)
print(f"Model response: {response}")
print(f"Latency:        {latency} ms")
print("\nSanity check passed." if response.strip().upper().startswith("B") else "\nModel responded but check the output above.")

Model response: B. Myocardial infarction.
Latency:        2626.7 ms

Sanity check passed.


# Section 4: Prompting Baselines

In this section I evaluate three prompting strategies on the validation set. All three use the same Mistral-7B-Instruct model with no modifications to the weights. The only thing that changes between them is how the question is presented to the model.

The three strategies are zero-shot prompting, few-shot prompting, and chain-of-thought prompting. I run all three on a 200-question subset of the validation set using a fixed random seed of 42, and record accuracy, Macro-F1, and inference latency for each. These results form the prompting baseline that RAG and QLoRA are compared against in Section 6.

## 4.1 Zero-Shot Prompting

Zero-shot prompting means asking the model a question directly with no examples or hints about how to answer. This is the simplest possible adaptation strategy and serves as the lowest baseline in the comparison. If a more complex strategy cannot beat zero-shot, it is not worth the added cost.

In [11]:
import re
from tqdm import tqdm

def format_zero_shot(row):
    prompt = f"""<s>[INST] You are a medical expert. Answer the following multiple choice question by choosing the single best answer.

Question: {row['question']}

A) {row['option_A']}
B) {row['option_B']}
C) {row['option_C']}
D) {row['option_D']}

Reply with only the letter of the correct answer (A, B, C, or D) followed by a brief 1-2 sentence justification. [/INST]"""
    return prompt


def extract_answer(response):
    response = response.strip().upper()
    match = re.match(r'^([ABCD])[^A-Z]', response)
    if match:
        return match.group(1)
    match = re.search(r'\b([ABCD])\b', response)
    if match:
        return match.group(1)
    if response and response[0] in 'ABCD':
        return response[0]
    return None


def run_prompting_experiment(df, prompt_formatter, max_new_tokens=150, desc="Running"):
    results = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        prompt = prompt_formatter(row)
        try:
            response, latency = generate_response(prompt, max_new_tokens=max_new_tokens)
            predicted = extract_answer(response)
        except Exception as e:
            response = ""
            latency = 0
            predicted = None
        results.append({
            'question':   row['question'],
            'answer':     row['answer'],
            'predicted':  predicted,
            'correct':    predicted == row['answer'],
            'latency_ms': latency,
            'response':   response
        })
        if (len(results)) % 100 == 0:
            so_far = sum(r['correct'] for r in results)
            print(f"  [{len(results)}/{len(df)}] Running accuracy: {so_far/len(results)*100:.1f}%")
    return pd.DataFrame(results)


print("Core functions defined: format_zero_shot, extract_answer, run_prompting_experiment")

Core functions defined: format_zero_shot, extract_answer, run_prompting_experiment


## 4.2 Zero-Shot Evaluation

I run zero-shot prompting on a 200-question subset sampled from the validation set using a fixed random seed of 42. This same 200-question subset is used across all five experimental conditions to ensure a fair comparison. For each question I record the predicted answer, whether it was correct, and the inference latency. I print a running accuracy update every 100 questions to monitor progress.

In [12]:
from tqdm import tqdm

# 200 questions is statistically sufficient for comparing techniques
# This is standard practice when working under compute constraints
EVAL_SAMPLE = 200

# Use a fixed seed so the same 200 questions are used across
# all five techniques for a fair comparison
eval_df = val_df.sample(n=EVAL_SAMPLE, random_state=42).reset_index(drop=True)

print(f"Evaluation subset: {len(eval_df)} questions (fixed seed=42)")
print("Running zero-shot experiment...\n")

zero_shot_results = run_prompting_experiment(
    eval_df,
    format_zero_shot,
    max_new_tokens=80,
    desc="Zero-Shot"
)

# Save results immediately
zero_shot_results.to_csv('/kaggle/working/zero_shot_results.csv', index=False)

# Summary
accuracy = zero_shot_results['correct'].mean() * 100
avg_latency = zero_shot_results['latency_ms'].mean()
null_predictions = zero_shot_results['predicted'].isna().sum()

print(f"\n=== ZERO-SHOT RESULTS ===")
print(f"Accuracy:           {accuracy:.2f}%")
print(f"Avg latency:        {avg_latency:.1f} ms")
print(f"Null predictions:   {null_predictions}")
print(f"Total questions:    {len(zero_shot_results)}")

Evaluation subset: 200 questions (fixed seed=42)
Running zero-shot experiment...



Zero-Shot:  50%|█████     | 100/200 [15:08<14:49,  8.90s/it]

  [100/200] Running accuracy: 34.0%


Zero-Shot: 100%|██████████| 200/200 [30:01<00:00,  9.01s/it]

  [200/200] Running accuracy: 35.5%

=== ZERO-SHOT RESULTS ===
Accuracy:           35.50%
Avg latency:        9005.4 ms
Null predictions:   6
Total questions:    200


## 4.3 Few-Shot Prompting

Few-shot prompting improves on zero-shot by providing the model with a small number of worked examples before the actual question. The idea is that seeing a few correctly answered questions helps the model understand the expected format and reasoning style. I use 3 examples drawn from the training set, chosen to cover different medical specialties so they are reasonably representative without being too long. The same 3 examples are used for every question in the evaluation set.

In [13]:
# Recreate eval_df from val_df using the same fixed seed used in all experiments
EVAL_SAMPLE = 200
eval_df = val_df.sample(n=EVAL_SAMPLE, random_state=42).reset_index(drop=True)
print(f"eval_df recreated: {len(eval_df)} questions")

eval_df recreated: 200 questions


In [14]:
# Select 3 fixed few-shot examples from training set
# Using fixed indices so results are reproducible
few_shot_examples = train_df.iloc[[10, 50, 100]]

def format_few_shot(row):
    """
    Formats a MedQA question as a few-shot prompt with 3 examples
    from the training set prepended before the actual question.
    """
    examples_text = ""
    for _, ex in few_shot_examples.iterrows():
        examples_text += f"""Question: {ex['question']}

A) {ex['option_A']}
B) {ex['option_B']}
C) {ex['option_C']}
D) {ex['option_D']}

Answer: {ex['answer']}) {ex['answer_text']}

"""

    prompt = f"""<s>[INST] You are a medical expert. Answer multiple choice questions by choosing the single best answer. Here are some examples of how to answer:

{examples_text}Now answer this question the same way:

Question: {row['question']}

A) {row['option_A']}
B) {row['option_B']}
C) {row['option_C']}
D) {row['option_D']}

Reply with only the letter of the correct answer (A, B, C, or D) followed by a brief 1-2 sentence justification. [/INST]"""
    return prompt


# Test on one example first before running the full set
sample_row = eval_df.iloc[0]
test_prompt = format_few_shot(sample_row)
response, latency = generate_response(test_prompt, max_new_tokens=80)
predicted = extract_answer(response)

print("=== FEW-SHOT TEST ===")
print("Response:", response)
print(f"Extracted: {predicted} | Correct: {sample_row['answer']} | Match: {predicted == sample_row['answer']}")
print(f"Latency: {latency} ms")

=== FEW-SHOT TEST ===
Response: A) Calcium gluconate: The patient's symptoms suggest possible cardiac dysrhythmia, and calcium gluconate is used to treat hypocalcemia, which can contribute to dysrhythmias. However, it's important to note that the definitive treatment for dysrhythmias depends on the specific type and may involve other medications or procedures
Extracted: A | Correct: C | Match: False
Latency: 17416.4 ms


## 4.4 Few-Shot Evaluation

I now run few-shot prompting across the same 200-question evaluation subset. The prompt is longer than zero-shot because it includes 3 training examples, which increases inference time per question. This trade-off between prompt length and accuracy is part of what the cost analysis in Section 6 will examine.

In [15]:
print("Running few-shot experiment on 200-question evaluation subset...")
print("This will take approximately 45-60 minutes due to longer prompts.\n")

few_shot_results = run_prompting_experiment(
    eval_df,
    format_few_shot,
    max_new_tokens=80,
    desc="Few-Shot"
)

# Save results immediately
few_shot_results.to_csv('/kaggle/working/few_shot_results.csv', index=False)

# Summary
accuracy = few_shot_results['correct'].mean() * 100
avg_latency = few_shot_results['latency_ms'].mean()
null_predictions = few_shot_results['predicted'].isna().sum()

print(f"\n=== FEW-SHOT RESULTS ===")
print(f"Accuracy:           {accuracy:.2f}%")
print(f"Avg latency:        {avg_latency:.1f} ms")
print(f"Null predictions:   {null_predictions}")
print(f"Total questions:    {len(few_shot_results)}")
# Verify file was actually saved
import os
size = os.path.getsize('/kaggle/working/few_shot_results.csv')
print(f"File saved successfully: {size} bytes")

Running few-shot experiment on 200-question evaluation subset...
This will take approximately 45-60 minutes due to longer prompts.



Few-Shot:  50%|█████     | 100/200 [25:56<26:16, 15.77s/it]

  [100/200] Running accuracy: 28.0%


Few-Shot: 100%|██████████| 200/200 [52:06<00:00, 15.63s/it]

  [200/200] Running accuracy: 32.0%

=== FEW-SHOT RESULTS ===
Accuracy:           32.00%
Avg latency:        15625.7 ms
Null predictions:   0
Total questions:    200
File saved successfully: 191942 bytes


## Recovery Cell

This cell reloads all previously completed experiment results from saved CSV files. If the Kaggle session timed out and I had to restart, I run this cell after reloading the model and function definitions to restore all results without rerunning the expensive experiments.

In [16]:
import os

# Create a persistent output directory
# Files saved here survive Kaggle session restarts
os.makedirs('/kaggle/working', exist_ok=True)
print("Working directory:", os.getcwd())
print("Files currently saved:", os.listdir('/kaggle/working'))

Working directory: /kaggle/working
Files currently saved: ['few_shot_results.csv', '__notebook__.ipynb', 'val.csv', 'test.csv', 'train.csv', 'zero_shot_results.csv', '=0.46.1']


In [17]:
import os

# Check all possible locations for saved results
paths_to_check = [
    '/kaggle/working',
    '/kaggle/working/results',
    '/tmp',
]

for path in paths_to_check:
    if os.path.exists(path):
        files = os.listdir(path)
        csv_files = [f for f in files if f.endswith('.csv')]
        print(f"\n{path}:")
        print(f"  CSV files: {csv_files}")


/kaggle/working:
  CSV files: ['few_shot_results.csv', 'val.csv', 'test.csv', 'train.csv', 'zero_shot_results.csv']

/tmp:
  CSV files: []


In [18]:
import pandas as pd
import os

# Kaggle persistent storage — files here survive session restarts
SAVE_DIR = '/kaggle/working'

results_store = {}

files = {
    'zero_shot':  '/kaggle/working/zero_shot_results.csv',
    'few_shot':   '/kaggle/working/few_shot_results.csv',
    'cot':        '/kaggle/working/cot_results.csv',
    'rag_k1':     '/kaggle/working/rag_k1_results.csv',
    'rag_k3':     '/kaggle/working/rag_k3_results.csv',
    'rag_k5':     '/kaggle/working/rag_k5_results.csv',
    'qlora':      '/kaggle/working/qlora_results.csv',
}

for name, fname in files.items():
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        results_store[name] = pd.read_csv(fpath)
        acc = results_store[name]['correct'].mean() * 100
        print(f"Loaded {name}: {len(results_store[name])} rows | Accuracy: {acc:.2f}%")
    else:
        print(f"Not yet available: {name}")

Loaded zero_shot: 200 rows | Accuracy: 35.50%
Loaded few_shot: 200 rows | Accuracy: 32.00%
Not yet available: cot
Not yet available: rag_k1
Not yet available: rag_k3
Not yet available: rag_k5
Not yet available: qlora


## 4.5 Chain-of-Thought Prompting

Chain-of-thought prompting asks the model to reason through a problem step by step before giving a final answer. The idea is that medical questions often require multi-step clinical reasoning, ruling out conditions, recalling drug mechanisms, connecting symptoms to diagnoses and forcing the model to write that reasoning out loud before committing to an answer tends to improve accuracy. I add a explicit instruction to think step by step and then state the final answer as the last line of the response.

In [19]:
def format_cot(row):
    """
    Formats a MedQA question as a chain-of-thought prompt.
    The model is explicitly asked to reason step by step before
    giving a final answer letter on the last line.
    """
    prompt = f"""<s>[INST] You are a medical expert. Answer the following multiple choice question by reasoning step by step.

Question: {row['question']}

A) {row['option_A']}
B) {row['option_B']}
C) {row['option_C']}
D) {row['option_D']}

Think through this step by step, then on the very last line write only: Answer: X where X is the letter of the correct option. [/INST]"""
    return prompt


def extract_cot_answer(response):
    """
    Extracts the final answer from a chain-of-thought response.
    Looks for 'Answer: X' pattern first, then falls back to
    the standard extract_answer function.
    """
    # Look for explicit Answer: X pattern at end of response
    match = re.search(r'Answer:\s*([ABCD])', response.upper())
    if match:
        return match.group(1)

    # Fallback to standard extractor
    return extract_answer(response)


# Test on one example before running full set
sample_row = eval_df.iloc[0]
test_prompt = format_cot(sample_row)
response, latency = generate_response(test_prompt, max_new_tokens=300)
predicted = extract_cot_answer(response)

print("=== COT TEST ===")
print("Response:", response)
print(f"\nExtracted: {predicted} | Correct: {sample_row['answer']} | Match: {predicted == sample_row['answer']}")
print(f"Latency: {latency} ms")

=== COT TEST ===
Response: Based on the given information, the patient's symptoms suggest possible cardiac causes for her syncope (fainting) and palpitations. However, there are no clear signs of life-threatening cardiac arrhythmias on her initial presentation. The electrocardiogram (ECG) will provide valuable information to help determine the specific arrhythmia, if any.

Given the available options, none of them are the first-line treatment for the patient's condition based on the information provided. The patient's symptoms could be indicative of various conditions, including anxiety, anemia, or cardiac causes. Further evaluation, including an ECG, is necessary to determine the underlying cause and guide appropriate treatment.

Answer: None of the above.

Extracted: B | Correct: C | Match: False
Latency: 17460.8 ms


## 4.6 Chain-of-Thought Evaluation

I now run chain-of-thought prompting across the same 200-question evaluation subset. CoT prompts generate longer responses than zero-shot or few-shot because the model writes out its reasoning before giving a final answer. This increases latency per question but the hypothesis is that the reasoning process improves accuracy on questions that require multi-step clinical logic.

In [20]:
print("Running chain-of-thought experiment on 200-question evaluation subset...")
print("This will take approximately 50-65 minutes due to longer responses.\n")

cot_results = run_prompting_experiment(
    eval_df,
    format_cot,
    max_new_tokens=300,
    desc="CoT"
)

# Save results immediately
cot_results.to_csv('/kaggle/working/cot_results.csv', index=False)

# Summary
accuracy = cot_results['correct'].mean() * 100
avg_latency = cot_results['latency_ms'].mean()
null_predictions = cot_results['predicted'].isna().sum()

print(f"\n=== CHAIN-OF-THOUGHT RESULTS ===")
print(f"Accuracy:           {accuracy:.2f}%")
print(f"Avg latency:        {avg_latency:.1f} ms")
print(f"Null predictions:   {null_predictions}")
print(f"Total questions:    {len(cot_results)}")

Running chain-of-thought experiment on 200-question evaluation subset...
This will take approximately 50-65 minutes due to longer responses.



CoT:  50%|█████     | 100/200 [42:31<49:34, 29.74s/it]

  [100/200] Running accuracy: 30.0%


CoT: 100%|██████████| 200/200 [1:24:13<00:00, 25.27s/it]

  [200/200] Running accuracy: 32.5%

=== CHAIN-OF-THOUGHT RESULTS ===
Accuracy:           32.50%
Avg latency:        25264.9 ms
Null predictions:   0
Total questions:    200


# Section 5: Retrieval-Augmented Generation (RAG)

The prompting results showed that zero-shot, few-shot, and chain-of-thought prompting all perform similarly around 32-35% accuracy on MedQA. This makes sense — Mistral-7B-Instruct is a general-purpose model and does not have deep enough medical knowledge encoded in its weights to reliably answer USMLE-level questions through prompting alone.

Retrieval-Augmented Generation addresses this by giving the model access to relevant information at inference time. Instead of relying purely on what the model already knows, RAG retrieves the most relevant passages from a document corpus and includes them in the prompt as context before asking the model to answer. The model can then ground its response in retrieved evidence rather than recalled knowledge.

My RAG pipeline works in two stages. First I build a FAISS vector database from the MedQA training set by encoding every training question and answer into dense vector embeddings. At inference time, for each validation question I retrieve the top-k most similar training examples and prepend them as context to the prompt. I then run this experiment with k set to 1, 3, and 5 to study how retrieval depth affects accuracy and latency — this is the ablation study described in my project proposal.

## 5.1 Building the FAISS Vector Database

In [21]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time

# Load the sentence transformer model for generating embeddings
# all-MiniLM-L6-v2 is lightweight, fast, and works well for semantic similarity
print("Loading sentence transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedder loaded.")

# Build corpus from training set
# Each entry combines the question with its correct answer
# so retrieved examples are informative for the validation question
print("\nBuilding corpus from training set...")
corpus = []
for _, row in train_df.iterrows():
    entry = f"Question: {row['question']} Answer: {row['answer']} - {row['answer_text']}"
    corpus.append(entry)

print(f"Corpus size: {len(corpus)} entries")

# Generate embeddings for the full training corpus
print("\nGenerating embeddings (this takes 3-5 minutes)...")
start = time.time()
corpus_embeddings = embedder.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
elapsed = round(time.time() - start, 1)
print(f"Embeddings generated in {elapsed}s")
print(f"Embedding shape: {corpus_embeddings.shape}")

# Build FAISS index
# IndexFlatIP uses inner product similarity — equivalent to cosine
# similarity when vectors are normalized
print("\nBuilding FAISS index...")
faiss.normalize_L2(corpus_embeddings)
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings)
print(f"FAISS index built with {index.ntotal} vectors")
print(f"Vector dimension: {dimension}")

# Record indexing time for cost analysis
indexing_time = elapsed
print(f"\nTotal indexing time: {indexing_time}s")

Loading sentence transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded.

Building corpus from training set...
Corpus size: 9160 entries

Generating embeddings (this takes 3-5 minutes)...


Batches:   0%|          | 0/144 [00:00<?, ?it/s]

Embeddings generated in 19.3s
Embedding shape: (9160, 384)

Building FAISS index...
FAISS index built with 9160 vectors
Vector dimension: 384

Total indexing time: 19.3s


## 5.2 RAG Retrieval and Prompt Formatting

With the FAISS index built, I now define the retrieval function and the RAG prompt formatter. For each validation question, I encode it into the same embedding space, query the FAISS index for the top-k most similar training examples, and prepend those examples as context to the prompt. I then run three separate experiments with k set to 1, 3, and 5 to measure how retrieval depth affects both accuracy and latency.

In [22]:
def retrieve_similar(question, k=3):
    """
    Retrieves the top-k most similar training examples
    for a given question using FAISS cosine similarity search.
    Returns a list of (question, answer) tuples.
    """
    query_embedding = embedder.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, k)

    retrieved = []
    for idx in indices[0]:
        row = train_df.iloc[idx]
        retrieved.append({
            'question':     row['question'],
            'option_A':     row['option_A'],
            'option_B':     row['option_B'],
            'option_C':     row['option_C'],
            'option_D':     row['option_D'],
            'answer':       row['answer'],
            'answer_text':  row['answer_text']
        })
    return retrieved


def format_rag(row, k=3):
    """
    Formats a MedQA question as a RAG prompt with k retrieved
    training examples prepended as context.
    """
    retrieved = retrieve_similar(row['question'], k=k)

    context = ""
    for i, ex in enumerate(retrieved):
        context += f"""Example {i+1}:
Question: {ex['question']}
A) {ex['option_A']}
B) {ex['option_B']}
C) {ex['option_C']}
D) {ex['option_D']}
Answer: {ex['answer']}) {ex['answer_text']}

"""

    prompt = f"""<s>[INST] You are a medical expert. Use the following similar examples as context to help answer the question below.

{context}Now answer this question:

Question: {row['question']}

A) {row['option_A']}
B) {row['option_B']}
C) {row['option_C']}
D) {row['option_D']}

Reply with only the letter of the correct answer (A, B, C, or D) followed by a brief 1-2 sentence justification. [/INST]"""
    return prompt


# Test retrieval on one example
sample_row = eval_df.iloc[0]

print("=== RETRIEVAL TEST (k=3) ===")
retrieved = retrieve_similar(sample_row['question'], k=3)
print(f"Query: {sample_row['question'][:100]}...")
print(f"\nRetrieved {len(retrieved)} examples:")
for i, r in enumerate(retrieved):
    print(f"  [{i+1}] {r['question'][:80]}... -> {r['answer']}")

# Test full RAG prompt on one example
response, latency = generate_response(
    format_rag(sample_row, k=3),
    max_new_tokens=100
)
predicted = extract_answer(response)

print(f"\n=== RAG RESPONSE (k=3) ===")
print("Response:", response)
print(f"\nExtracted: {predicted} | Correct: {sample_row['answer']} | Match: {predicted == sample_row['answer']}")
print(f"Latency: {latency} ms")

=== RETRIEVAL TEST (k=3) ===
Query: A 28-year-old woman is brought to the emergency department by a friend after fainting at work and hi...

Retrieved 3 examples:
  [1] A 57-year-old man is brought to the emergency department by his son for odd beha... -> D
  [2] A 49-year-old man seeks evaluation at an urgent care clinic with a complaint of ... -> A
  [3] A 67-year-old woman comes to the emergency department 1 hour after her husband s... -> B

=== RAG RESPONSE (k=3) ===
Response: Based on the information provided, it is not clear if the patient's symptoms are related to a cardiac condition that would require the use of calcium gluconate, flecainide, magnesium sulfate, or procainamide. The patient's symptoms of fainting, palpitations, shortness of breath, nausea, and chest pain are non-specific and could be due to various causes, including orthostatic hypotension,

Extracted: A | Correct: C | Match: False
Latency: 17766.3 ms


## 5.3 RAG Experiments: Ablation over k

I run three separate RAG experiments using k=1, k=3, and k=5 retrieved examples. This ablation study measures how retrieval depth affects both accuracy and inference latency. A higher k gives the model more context but also makes the prompt longer and slower. The goal is to find whether more retrieved examples actually help or whether they add noise that hurts performance.

In [23]:
import functools

for k in [1, 3, 5]:
    print(f"\nRunning RAG experiment with k={k}...")
    print(f"Estimated time: {'20-25' if k==1 else '30-40' if k==3 else '40-50'} minutes\n")

    # Create a prompt formatter with k fixed using functools.partial
    rag_formatter = functools.partial(format_rag, k=k)

    rag_results = run_prompting_experiment(
        eval_df,
        rag_formatter,
        max_new_tokens=100,
        desc=f"RAG k={k}"
    )

    # Save immediately after each k
    rag_results.to_csv(f'/kaggle/working/rag_k{k}_results.csv', index=False)

    accuracy = rag_results['correct'].mean() * 100
    avg_latency = rag_results['latency_ms'].mean()
    null_predictions = rag_results['predicted'].isna().sum()

    print(f"\n=== RAG k={k} RESULTS ===")
    print(f"Accuracy:         {accuracy:.2f}%")
    print(f"Avg latency:      {avg_latency:.1f} ms")
    print(f"Null predictions: {null_predictions}")
    print(f"Total questions:  {len(rag_results)}")
    print("-" * 40)

print("\nAll RAG experiments complete.")


Running RAG experiment with k=1...
Estimated time: 20-25 minutes



RAG k=1:  50%|█████     | 100/200 [19:24<21:01, 12.61s/it]

  [100/200] Running accuracy: 40.0%


RAG k=1: 100%|██████████| 200/200 [38:33<00:00, 11.57s/it]


  [200/200] Running accuracy: 38.0%

=== RAG k=1 RESULTS ===
Accuracy:         38.00%
Avg latency:      11555.1 ms
Null predictions: 2
Total questions:  200
----------------------------------------

Running RAG experiment with k=3...
Estimated time: 30-40 minutes



...

  [200/200] Running accuracy: 34.0%

=== RAG k=3 RESULTS ===
Accuracy:         34.00%
Avg latency:      14880.5 ms
Null predictions: 2
Total questions:  200
----------------------------------------

Running RAG experiment with k=5...
Estimated time: 40-50 minutes



RAG k=5:  50%|█████     | 100/200 [31:53<30:23, 18.24s/it]

  [100/200] Running accuracy: 30.0%


RAG k=5: 100%|██████████| 200/200 [1:02:50<00:00, 18.85s/it]

  [200/200] Running accuracy: 33.5%

=== RAG k=5 RESULTS ===
Accuracy:         33.50%
Avg latency:      18833.3 ms
Null predictions: 4
Total questions:  200
----------------------------------------

All RAG experiments complete.


# Section 6: QLoRA Fine-Tuning

The prompting and RAG experiments established a ceiling of around 38% accuracy using inference-only adaptation strategies. To go beyond this, the model needs to actually learn from the medical training data rather than just retrieving or being shown examples at inference time.

QLoRA (Quantized Low-Rank Adaptation) makes this feasible under limited compute. Instead of updating all 7 billion parameters of Mistral-7B, QLoRA keeps the base model frozen in 4-bit quantization and trains a small set of low-rank adapter matrices that are injected into the attention layers. The number of trainable parameters is a tiny fraction of the full model, which means training fits comfortably within the GPU memory available on Kaggle.

I fine-tune on the MedQA training set by formatting each training example as an instruction-following prompt with the correct answer and a brief justification. After training I merge the LoRA adapters back into the base model and evaluate on the same 200-question evaluation subset used by all other techniques.

## 6.1 Fine-Tuning Setup

In [24]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

# Prepare model for QLoRA training
# This adds gradient checkpointing and prepares the quantized
# model to accept trainable LoRA adapter layers
print("Preparing model for QLoRA training...")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# LoRA configuration
# r=16 is the rank of the adapter matrices — higher rank means
# more parameters but better expressiveness
# target_modules specifies which layers get LoRA adapters
# alpha=32 is the scaling factor for the adapter outputs
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA adapters to the model
model = get_peft_model(model, lora_config)

# Print trainable parameter count to confirm QLoRA is working
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.4f}%")

Preparing model for QLoRA training...
Trainable parameters: 41,943,040
Total parameters:     3,794,014,208
Trainable %:          1.1055%


## 6.2 Preparing the Training Data

I format each training example as an instruction, following prompt that matches the same format used during inference. Each example includes the question, the four answer options, and the correct answer with a brief explanation. This consistency between training and inference format is important, the model learns to produce outputs in exactly the style we evaluate it on.

In [25]:
from datasets import Dataset

def format_training_example(row):
    """
    Formats a training example as a complete instruction-response
    pair for supervised fine-tuning. The model learns to map
    the question prompt to the correct answer and justification.
    """
    prompt = f"""<s>[INST] You are a medical expert. Answer the following multiple choice question by choosing the single best answer.

Question: {row['question']}

A) {row['option_A']}
B) {row['option_B']}
C) {row['option_C']}
D) {row['option_D']}

Reply with only the letter of the correct answer (A, B, C, or D) followed by a brief 1-2 sentence justification. [/INST]"""

    response = f" {row['answer']}) {row['answer_text']}</s>"

    return prompt + response


# Format all training examples
print("Formatting training examples...")
formatted_examples = [format_training_example(row) for _, row in train_df.iterrows()]

print(f"Total training examples: {len(formatted_examples)}")
print(f"\n=== SAMPLE FORMATTED EXAMPLE ===")
print(formatted_examples[0])

# Convert to HuggingFace Dataset format required by SFTTrainer
train_dataset = Dataset.from_dict({"text": formatted_examples})
print(f"\nTraining dataset created: {len(train_dataset)} examples")

Formatting training examples...
Total training examples: 9160

=== SAMPLE FORMATTED EXAMPLE ===
<s>[INST] You are a medical expert. Answer the following multiple choice question by choosing the single best answer.

Question: A 60-year-old man comes to the physician because of flank pain, rash, and blood-tinged urine for 1 day. Two months ago, he was started on hydrochlorothiazide for hypertension. He takes acetaminophen for back pain. Examination shows a generalized, diffuse maculopapular rash. Serum studies show a creatinine concentration of 3.0 mg/dL. Renal ultrasonography shows no abnormalities. Which of the following findings is most likely to be observed in this patient?

A) Dermal IgA deposition on skin biopsy
B) Crescent-shape extracapillary cell proliferation
C) Mesangial IgA deposits on renal biopsy
D) Urinary eosinophils

Reply with only the letter of the correct answer (A, B, C, or D) followed by a brief 1-2 sentence justification. [/INST] D) Urinary eosinophils</s>

Trainin

## 6.3 Training

I train the LoRA adapters using the SFTTrainer from the TRL library. Due to compute constraints on Kaggle's free GPU tier, I train for 1 epoch on a 2000-example subset of the training data. The learning rate is set to 2e-4 which is the standard recommended value for QLoRA fine-tuning. I use gradient accumulation to simulate a larger effective batch size without increasing memory usage. The training loss decreased steadily from 1.33 to 0.89 over 125 steps, confirming the model was learning from the medical training data.

In [26]:
import time
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Using 2000 examples for 1 epoch to keep training time manageable
# This is sufficient to demonstrate domain adaptation for comparison purposes
sample_df = train_df.sample(n=2000, random_state=42).reset_index(drop=True)
formatted_sample = [format_training_example(row) for _, row in sample_df.iterrows()]
train_dataset_small = Dataset.from_dict({"text": formatted_sample})

print(f"Training on {len(train_dataset_small)} examples for 1 epoch")

training_args = SFTConfig(
    output_dir="./qlora_output",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=25,
    save_strategy="no",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=20,
    report_to="none",
    dataloader_num_workers=0,
    max_length=512,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_small,
    processing_class=tokenizer,
)

print("Starting QLoRA training (1 epoch, 2000 examples)...")
print("Expected time: 45-60 minutes\n")

training_start = time.time()
trainer.train()
training_time = round(time.time() - training_start, 1)

print(f"\nTraining complete.")
print(f"Total training time: {training_time}s ({round(training_time/60, 1)} minutes)")

print(f"\nSaving trained model...")
trainer.save_model("./qlora_trained")
tokenizer.save_pretrained("./qlora_trained")
print("Model saved to ./qlora_trained")

Training on 2000 examples for 1 epoch


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting QLoRA training (1 epoch, 2000 examples)...
Expected time: 45-60 minutes



Step,Training Loss
25,1.330304
50,0.979049
75,0.918429
100,0.915223
125,0.893366



Training complete.
Total training time: 19902.6s (331.7 minutes)

Saving trained model...
Model saved to ./qlora_trained


In [27]:
import shutil
import os

# Copy all results to /kaggle/working which persists after session ends
files_to_save = [
    'zero_shot_results.csv',
    'few_shot_results.csv', 
    'cot_results.csv',
    'rag_k1_results.csv',
    'rag_k3_results.csv',
    'rag_k5_results.csv',
    'qlora_results.csv',
]

print("Verifying saved files:")
for f in files_to_save:
    path = f'/kaggle/working/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  {f}: {size} bytes")
    else:
        print(f"  {f}: NOT FOUND")

Verifying saved files:
  zero_shot_results.csv: 201034 bytes
  few_shot_results.csv: 191942 bytes
  cot_results.csv: 332166 bytes
  rag_k1_results.csv: 209736 bytes
  rag_k3_results.csv: 205889 bytes
  rag_k5_results.csv: 205372 bytes
  qlora_results.csv: NOT FOUND


## Note

This notebook covers Sections 1 through 6, including all prompting experiments, RAG pipeline, and QLoRA fine-tuning. The QLoRA model evaluation, full results analysis in Section 7, and conclusion in Section 8 are contained in the analysis notebook due to Kaggle's 12-hour session timeout constraint. All experiment results saved here as CSV files are loaded and analyzed in the analysis notebook.